In [ ]:
# Install all required packages:
# - langchain: core framework
# - langchain-chroma: integration with Chroma, a local vector database used to store and search embeddings
# - langchain-community: community loaders (DirectoryLoader, WikipediaRetriever, etc.)
# - langchain-openai: OpenAI model + embedding support
# - langchain-text-splitters: tools for splitting long documents into smaller chunks
# - wikipedia: Python client used by WikipediaRetriever
!pip install -q langchain langchain-chroma langchain-community langchain-openai langchain-text-splitters wikipedia


In [ ]:
from google.colab import userdata
# batched: splits an iterable into fixed-size batches (used when adding many documents to a vector store)
from itertools import batched
# create_agent: builds a reasoning agent that can use tools automatically
from langchain.agents import create_agent
from langchain.messages import HumanMessage
# Chroma: an in-process vector database; stores document embeddings for semantic search
from langchain_chroma import Chroma
# DirectoryLoader: loads all matching files from a local folder
# TextLoader: reads plain text files
from langchain_community.document_loaders import DirectoryLoader, TextLoader
# WikipediaRetriever: fetches Wikipedia articles as LangChain Documents
from langchain_community.retrievers import WikipediaRetriever
from langchain_core.messages import BaseMessage
# PromptTemplate: formats a string template with variable substitution
from langchain_core.prompts import PromptTemplate
# create_retriever_tool: wraps a retriever so the agent can call it as a tool
from langchain_core.tools import create_retriever_tool
from langchain_openai import ChatOpenAI
# RecursiveCharacterTextSplitter: splits text by trying paragraph breaks, then sentence breaks, then word breaks
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()


You can download books in .txt format from the [Gutenberg Project](https://www.gutenberg.org/browse/scores/top).

In [ ]:
# Load all .txt book files from the /content/books directory.
# DirectoryLoader walks the folder and creates one LangChain Document per file.
# loader_cls=TextLoader tells it to use the plain-text loader (not the default unstructured loader).
# NOTE: By default, if `loader_cls` is not provided, the `UnstructuredFileLoader` will be used (which requires installing the `unstructured` package in addition).
directory_loader = DirectoryLoader("/content/books", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
books = directory_loader.load()


In [ ]:
# Split each book into smaller overlapping chunks.
# LLMs and embedding models have a limited context window, so large documents must be chunked.
# - chunk_size=1000: each chunk is at most 1000 characters
# - chunk_overlap=200: consecutive chunks share 200 characters to avoid cutting context at boundaries
recursive_text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_book_chunks = recursive_text_splitter.split_documents(books)


In [ ]:
# Show how many books were loaded and how many chunks they were split into.
# A single book can produce hundreds of chunks, depending on its length.
print(f"Originally, there were {len(books)} books that were then split into {len(split_book_chunks)} chunks.")


In [ ]:
# Create a Chroma vector store and populate it with the book chunks.
# Chroma stores each chunk as a numerical embedding (a vector) so it can be searched semantically.
# persist_directory="/content/chroma" saves the database to disk so it survives kernel restarts.
vector_store = Chroma(collection_name="books", persist_directory="/content/chroma")

# Add documents in batches of 256 to avoid memory issues with very large datasets.
# NOTE: As there are too many chunks, add them in batches.
for index, batch in enumerate(batched(split_book_chunks, 256)):
    vector_store.add_documents(batch)
    print(f"Processed batch #{index + 1}")


In [ ]:
# Convert the vector store into a retriever.
# A retriever is a simpler interface that accepts a query and returns the most relevant documents.
# Under the hood it computes the embedding of the query and finds the closest stored chunks.
quote_retriever = vector_store.as_retriever()


In [ ]:
# Test the retriever with a sample query.
# It will return the book chunks whose content is semantically closest to the query string,
# even if the exact words don't match (because it uses embeddings, not keyword search).
quote_retriever.invoke(input="A Day Without Laughter Is a Day Wasted")


In [ ]:
# Create a Wikipedia retriever to fetch background info on authors and books.
# - top_k_results=5: retrieve the 5 most relevant Wikipedia articles
# - doc_content_chars_max=0: fetch only the article summaries, not the full page text
#   (summaries are enough for context, and keeping them short saves tokens)
# NOTE: We will use the summary only, so there is no need to fetch the entire Wikipedia page.
wikipedia_retriever = WikipediaRetriever(top_k_results=5, doc_content_chars_max=0)


In [ ]:
# Test the Wikipedia retriever — it will return summary Documents for articles matching "Anna Karenina"
wikipedia_retriever.invoke(input="Anna Karenina")


In [ ]:
# Build a literary assistant agent that uses both retrievers as tools.
# create_retriever_tool wraps a retriever so the agent can call it by name during reasoning.
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key),
    tools=[
        create_retriever_tool(
            quote_retriever,               # The Chroma vector store retriever
            name="lookup_quote",           # The tool name the agent uses to call it
            description="Search the book database for semantically relevant quotes.",
            document_separator="\n=====\n",  # How multiple retrieved documents are joined
            # PromptTemplate formats each retrieved document before sending it to the model
            document_prompt=PromptTemplate(input_variables=["source", "page_content"], template="Source: {source}\n\n{page_content}")
        ),
        create_retriever_tool(
            wikipedia_retriever,           # The Wikipedia retriever
            name="lookup_info",
            description="""
                        Search for information about authors or books.
                        Always use this tool after finding a quote to provide context about the author or the book.
                        The query MUST be a short keyword like an author name or book title, NOT a full sentence.
                        """,
            document_separator="\n=====\n",
            document_prompt=PromptTemplate(input_variables=["title", "summary"], template="Article: {title}\n\n{summary}")
        )
    ],
    # System prompt instructs the agent to always look up quotes and enrich them with Wikipedia context
    system_prompt="""
                  You are an expert literary assistant with deep knowledge of world literature, poetry, and prose.
                  Use the \"lookup_quote\" tool for every interaction with the user to find relevant quotes.
                  Then, extend your knowledge (for the ones you like the most) using the \"lookup_info\" tool.
                  """
)


In [ ]:
# Ask the literary agent a philosophical question.
# It will:
#   1. Use lookup_quote to find relevant passages from the loaded books
#   2. Use lookup_info to fetch Wikipedia context about the authors/works it found
#   3. Compose a thoughtful answer grounded in real literature
meaning_of_love = agent.invoke(
    input={
        "messages": [HumanMessage("Is love a moral force or simply a powerful emotion?")]
    }
)


In [ ]:
# Print the full conversation including all tool calls and the agent's final literary response.
print_conversation(meaning_of_love["messages"])
